# 01 — Ingestão de Dados
**Projeto:** Previsão do PLD Mensal — Submercado SE/CO | **Período:** 2015–2024

Cada seção gera/coleta uma fonte. Linhas comentadas = coleta real dos portais oficiais.

## 0. Setup

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

BASE_DIR = Path().resolve().parent
RAW_DIR  = BASE_DIR / 'data' / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)

SUBMERCADO = 'SE'
np.random.seed(42)

datas = pd.date_range('2015-01-01', '2024-12-31', freq='MS')
N   = len(datas)
T   = np.arange(N, dtype=float)
MES = datas.month.to_numpy(dtype=float)   # numpy array — evita problemas de Index

print(f"Período: {datas[0].strftime('%b/%Y')} → {datas[-1].strftime('%b/%Y')} ({N} meses)")
print(f"Raw dir: {RAW_DIR}")


Período: Jan/2015 → Dec/2024 (120 meses)
Raw dir: /home/claude/pld_forecast/data/raw


## 1. PLD Mensal — CCEE
**Portal:** https://dadosabertos.ccee.org.br/dataset/pld_media_mensal

In [2]:
# ── COLETA REAL ───────────────────────────────────────────────────────────
# df_pld = pd.read_csv(RAW_DIR / 'PLD_MEDIA_MENSAL.csv', sep=';', decimal=',',
#                      parse_dates=['din_referencia'])
# df_pld = df_pld[df_pld['nom_submercado'] == SUBMERCADO][['din_referencia','val_pld']]

# ── SIMULAÇÃO (reproduz padrão real: sazonalidade, crise 2021, tendência) ──
pld_vals = np.clip(
    180
    + (-40) * np.cos(2 * np.pi * (MES - 1) / 12)   # sazonalidade
    + 0.8 * T                                         # tendência
    + 25  * np.sin(2 * np.pi * T / 48)               # ciclo hidrológico 4a
    + np.where((datas.year == 2021) & (datas.month >= 6), 180, 0)  # crise
    + np.random.normal(0, 18, N),
    50, 900)

df_pld = pd.DataFrame({'din_referencia': datas, 'nom_submercado': SUBMERCADO,
                        'val_pld': pld_vals.round(2)})
df_pld.to_parquet(RAW_DIR / 'pld_mensal_raw.parquet', index=False)
df_pld.to_csv(RAW_DIR / 'pld_mensal_raw.csv', index=False)
print(f"PLD: {len(df_pld)} registros | média={pld_vals.mean():.1f} | min={pld_vals.min():.1f} | max={pld_vals.max():.1f}")


PLD: 120 registros | média=239.9 | min=114.9 | max=446.4


## 2. EAR % — ONS
**Portal:** https://dados.ons.org.br/dataset/ear-diario-por-subsistema

In [3]:
# ── COLETA REAL ───────────────────────────────────────────────────────────
# df_ear_d = pd.read_csv(RAW_DIR / 'EAR_Diario_por_Subsistema.csv', sep=';', decimal=',',
#                        parse_dates=['din_referencia'])
# df_ear = (df_ear_d[df_ear_d['nom_subsistema'] == 'SE']
#           .assign(din_referencia=lambda d: d['din_referencia'].dt.to_period('M').dt.to_timestamp())
#           .groupby('din_referencia', as_index=False)['val_ear_pct'].mean()
#           .rename(columns={'val_ear_pct': 'ear_pct_se'}))

# ── SIMULAÇÃO ─────────────────────────────────────────────────────────────
ear_vals = np.clip(
    55
    + 18 * np.cos(2 * np.pi * (MES - 1) / 12)         # sazonal (cheio fev/mar)
    - 0.15 * T                                           # tendência queda
    + np.where((datas.year == 2021) & (datas.month >= 5), -20, 0).astype(float)
    + np.random.normal(0, 5, N),
    5, 100)

df_ear = pd.DataFrame({'din_referencia': datas, 'ear_pct_se': ear_vals.round(2)})
df_ear.to_parquet(RAW_DIR / 'ear_mensal_raw.parquet', index=False)
df_ear.to_csv(RAW_DIR / 'ear_mensal_raw.csv', index=False)
print(f"EAR: {len(df_ear)} registros | média={ear_vals.mean():.1f}%")


EAR: 120 registros | média=45.1%


## 3. ENA — ONS
**Portal:** https://dados.ons.org.br/dataset/ena-diario-por-subsistema

In [4]:
# ── COLETA REAL ───────────────────────────────────────────────────────────
# df_ena_d = pd.read_csv(RAW_DIR / 'ENA_Diario_por_Subsistema.csv', sep=';', decimal=',',
#                        parse_dates=['din_referencia'])
# df_ena = (df_ena_d[df_ena_d['nom_subsistema'] == 'SE']
#           .assign(din_referencia=lambda d: d['din_referencia'].dt.to_period('M').dt.to_timestamp())
#           .groupby('din_referencia', as_index=False)['val_enabrutamwmed'].mean()
#           .rename(columns={'val_enabrutamwmed': 'ena_mwmed_se'}))

# ── SIMULAÇÃO ─────────────────────────────────────────────────────────────
ena_vals = np.clip(
    28000
    + 9000 * np.cos(2 * np.pi * (MES - 2) / 12)       # pico dez/jan
    + np.random.normal(0, 2500, N),
    5000, 70000)

df_ena = pd.DataFrame({'din_referencia': datas, 'ena_mwmed_se': ena_vals.round(0)})
df_ena.to_parquet(RAW_DIR / 'ena_mensal_raw.parquet', index=False)
df_ena.to_csv(RAW_DIR / 'ena_mensal_raw.csv', index=False)
print(f"ENA: {len(df_ena)} registros | média={ena_vals.mean():.0f} MWmed")


ENA: 120 registros | média=28024 MWmed


## 4. Geração Térmica — ONS
**Portal:** https://dados.ons.org.br/dataset/geracao-termica-motivo-despacho

In [5]:
# ── COLETA REAL ───────────────────────────────────────────────────────────
# df_term_d = pd.read_csv(RAW_DIR / 'Geracao_Termica_Motivo_Despacho.csv', sep=';', decimal=',',
#                         parse_dates=['din_referencia'])
# df_term = (df_term_d[df_term_d['nom_subsistema'] == 'SE']
#            .assign(din_referencia=lambda d: d['din_referencia'].dt.to_period('M').dt.to_timestamp())
#            .groupby('din_referencia', as_index=False)['val_gerpot'].sum()
#            .rename(columns={'val_gerpot': 'geracao_termica_mwmed'}))

# ── SIMULAÇÃO ─────────────────────────────────────────────────────────────
term_vals = np.clip(
    8000
    - 2500 * np.cos(2 * np.pi * (MES - 1) / 12)       # mais despacho no seco
    + np.where((datas.year == 2021) & (datas.month >= 5), 5000, 0).astype(float)
    + np.random.normal(0, 800, N),
    1000, 25000)

df_term = pd.DataFrame({'din_referencia': datas, 'geracao_termica_mwmed': term_vals.round(0)})
df_term.to_parquet(RAW_DIR / 'termica_mensal_raw.parquet', index=False)
df_term.to_csv(RAW_DIR / 'termica_mensal_raw.csv', index=False)
print(f"Térmica: {len(df_term)} registros | média={term_vals.mean():.0f} MWmed")


Térmica: 120 registros | média=8383 MWmed


## 5. Precipitação — INMET
**Portal:** https://bdmep.inmet.gov.br/

In [6]:
# ── COLETA REAL ───────────────────────────────────────────────────────────
# df_precip = pd.read_csv(RAW_DIR / 'precipitacao_se.csv', sep=';', decimal=',',
#                         parse_dates=['din_referencia'])
# df_precip = (df_precip
#              .assign(din_referencia=lambda d: d['din_referencia'].dt.to_period('M').dt.to_timestamp())
#              .groupby('din_referencia', as_index=False)['val_precipitacao'].mean())

# ── SIMULAÇÃO ─────────────────────────────────────────────────────────────
precip_vals = np.clip(
    110
    + 85 * np.cos(2 * np.pi * (MES - 1) / 12)         # chuvas nov–mar
    + np.abs(np.random.normal(0, 20, N)),
    10, 350)

df_precip = pd.DataFrame({'din_referencia': datas, 'precipitacao_mm': precip_vals.round(1)})
df_precip.to_parquet(RAW_DIR / 'precipitacao_mensal_raw.parquet', index=False)
df_precip.to_csv(RAW_DIR / 'precipitacao_mensal_raw.csv', index=False)
print(f"Precipitação: {len(df_precip)} registros | média={precip_vals.mean():.1f} mm")


Precipitação: 120 registros | média=125.2 mm


## 6. Validação Final

In [7]:
arquivos = {
    'PLD Mensal':      'pld_mensal_raw.parquet',
    'EAR %':           'ear_mensal_raw.parquet',
    'ENA':             'ena_mensal_raw.parquet',
    'Geração Térmica': 'termica_mensal_raw.parquet',
    'Precipitação':    'precipitacao_mensal_raw.parquet',
}
print(f"{'Dataset':<20} {'N':>5} {'Início':>10} {'Fim':>10} {'Nulls':>6}")
print("-" * 55)
for nome, fname in arquivos.items():
    df = pd.read_parquet(RAW_DIR / fname)
    col = [c for c in df.columns if c not in ('din_referencia','nom_submercado')][0]
    print(f"{nome:<20} {len(df):>5} "
          f"{str(df['din_referencia'].min())[:7]:>10} "
          f"{str(df['din_referencia'].max())[:7]:>10} "
          f"{df[col].isna().sum():>6}")
print("\n✅ Todos os arquivos raw prontos → prosseguir para notebook 02.")


Dataset                  N     Início        Fim  Nulls
-------------------------------------------------------
PLD Mensal             120    2015-01    2024-12      0
EAR %                  120    2015-01    2024-12      0
ENA                    120    2015-01    2024-12      0
Geração Térmica        120    2015-01    2024-12      0
Precipitação           120    2015-01    2024-12      0

✅ Todos os arquivos raw prontos → prosseguir para notebook 02.
